IMPORT ET CONFIGURATION

In [1]:
!pip install mne

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration professionnelle
mne.set_log_level('WARNING')  # Évite les messages superflus
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Création des dossiers de sortie
Path("../figures").mkdir(exist_ok=True)
Path("../reports").mkdir(exist_ok=True)

print("✅ Bibliothèques chargées")
print(f"MNE version: {mne.__version__}")

✅ Bibliothèques chargées
MNE version: 1.11.0


CHARGEMENT D'UN SUJET EXEMPLE

In [2]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration professionnelle
mne.set_log_level('WARNING')  # Évite les messages superflus
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Création des dossiers de sortie
Path("../figures").mkdir(exist_ok=True)
Path("../reports").mkdir(exist_ok=True)

print("✅ Bibliothèques chargées")
print(f"MNE version: {mne.__version__}")

✅ Bibliothèques chargées
MNE version: 1.11.0


In [3]:
print("\n" + "="*60)
print("CHARGEMENT DES DONNÉES PHYSIONET")
print("="*60)

# Choix des sujets (on commence par 1 sujet pour tester)
subjects = [1, 2, 3]  # Chargement des 3 sujets requis

# Runs d'intérêt : 6 (mouvement réel), 10 (imagerie), 14 (imagerie avec repos)
runs = [6, 10, 14]

# Dictionnaire pour stocker les objets Raw par sujet
raw_data = {}

for subject in subjects:
    print(f"\n📂 Chargement du sujet {subject}...")
    
    # Étape 1 : Récupérer les chemins des fichiers EDF
    raw_fnames = mne.datasets.eegbci.load_data(subject, runs, verbose=False)
    
    # Étape 2 : Lire chaque fichier EDF et les concaténer
    raws = []
    for fname in raw_fnames:
        raw = mne.io.read_raw_edf(fname, preload=True, verbose=False)
        raws.append(raw)
    
    # Concaténation temporelle
    raw_concat = mne.concatenate_raws(raws, verbose=False)
    
    # Étape 3 : Standardiser les noms des canaux
    mne.datasets.eegbci.standardize(raw_concat)
    
    # Étape 4 : Ajouter le montage des électrodes
    montage = mne.channels.make_standard_montage('standard_1020')
    raw_concat.set_montage(montage, verbose=False)
    
    raw_data[subject] = raw_concat
    
    print(f"   ✅ Sujet {subject} chargé : {raw_concat.times[-1]:.1f} secondes")
    print(f"   📊 Canaux EEG: {len(raw_concat.ch_names)}")
    print(f"   🎵 Fréquence: {raw_concat.info['sfreq']} Hz")

print("\n✅ Chargement terminé !")
print(f"🔑 Clés disponibles dans raw_data : {list(raw_data.keys())}")



CHARGEMENT DES DONNÉES PHYSIONET

📂 Chargement du sujet 1...


StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.

In [ ]:
# ============================================================================
# EXPLICATION LIGNE PAR LIGNE
# ============================================================================
# mne.datasets.eegbci.load_data() : télécharge depuis PhysioNet (1er appel)
# raw_fnames : liste de chemins ['S001R06.edf', 'S001R10.edf', 'S001R14.edf']
# mne.io.read_raw_edf() : parse le format EDF et crée un objet Raw
# preload=True : charge tout le signal en RAM (nécessaire pour filtrage)
# mne.concatenate_raws() : fusionne plusieurs Raw temporellement
# standardize() : renomme 'EEG C3' -> 'C3'
# set_montage() : attache les coordonnées 3D des électrodes

# ============================================================================
# 3. INSPECTION DES MÉTADONNÉES
# ============================================================================


INSPECTION DES MÉTADONNÉES

In [ ]:
print("\n" + "="*60)
print("INSPECTION DES MÉTADONNÉES - Sujet 1")
print("="*60)

raw_s1 = raw_data[1]

print("\n📋 Informations générales :")
print(f"   - Durée totale : {raw_s1.times[-1]:.1f} secondes ({raw_s1.times[-1]/60:.1f} minutes)")
print(f"   - Points temporels : {raw_s1.n_times:,}")
print(f"   - Canaux EEG : {len(mne.pick_types(raw_s1.info, meg=False, eeg=True))}")
print(f"   - Canaux EOG : {len(mne.pick_types(raw_s1.info, meg=False, eog=True))}")
print(f"   - Mémoire utilisée : ~{raw_s1.n_times * len(raw_s1.ch_names) * 8 / 1e6:.1f} MB")

# Afficher les 15 premiers canaux
print("\n📊 Liste des 15 premiers canaux EEG :")
for i, ch in enumerate(raw_s1.ch_names[:15]):
    print(f"   {i+1:2d}. {ch}")


In [ ]:
# ============================================================================
# EXPLICATION : Pourquoi ces métadonnées sont importantes ?
# ============================================================================
# - Durée : 3 runs × ~2 min = 6 min de signal continu
# - 160 Hz : échantillonnage suffisant pour voir jusqu'à 80 Hz (théorème Nyquist)
# - 64 canaux : bonne résolution spatiale pour localiser les sources motrices

ANALYSE DES ANNOTATIONS (ÉVÉNEMENTS)

In [ ]:
print("\n" + "="*60)
print("ANALYSE DES ÉVÉNEMENTS")
print("="*60)

# Extraire les événements depuis le champ annotations
# Explication : Les annotations sont des marqueurs temporels encodés dans le fichier
events, event_id = mne.events_from_annotations(raw_s1, verbose=False)

print("\n🏷️  Dictionnaire des événements :")
for event_name, event_code in event_id.items():
    print(f"   {event_code} -> {event_name}")

print(f"\n🎯 Nombre total d'événements : {len(events)}")

# Compter les occurrences de chaque type
event_counts = {}
for event_code in np.unique(events[:, 2]):
    count = np.sum(events[:, 2] == event_code)
    event_name = [k for k, v in event_id.items() if v == event_code][0]
    event_counts[event_name] = count

print("\n📊 Distribution des événements :")
for event_name, count in event_counts.items():
    print(f"   - {event_name}: {count} occurrences")

# Créer un DataFrame pandas pour une analyse plus poussée
event_df = pd.DataFrame(events, columns=['sample', 'padding', 'code'])
event_df['time_sec'] = event_df['sample'] / raw_s1.info['sfreq']
event_df['event_name'] = event_df['code'].map({v: k for k, v in event_id.items()})

print("\n⏱️  Exemple des 10 premiers événements :")
print(event_df[['time_sec', 'event_name']].head(10))

# Calculer les intervalles entre événements
event_df['time_diff'] = event_df['time_sec'].diff()
print(f"\n⏱️  Intervalle moyen entre événements : {event_df['time_diff'].mean():.2f} secondes")


In [ ]:
# ============================================================================
# INTERPRÉTATION SCIENTIFIQUE
# ============================================================================
# Les annotations T1 (main gauche), T2 (main droite) sont les événements
# d'intérêt. On observe typiquement 90-100 événements par classe sur 6 min.
# L'intervalle ~4 secondes correspond au protocole expérimental :
# - 2 secondes de préparation
# - 2 secondes d'imagerie
# - Repos aléatoire


VISUALISATION DU SIGNAL BRUT

In [ ]:
print("\n" + "="*60)
print("VISUALISATION")
print("="*60)

# 5.1 Signal temporel sur 10 secondes
fig1 = raw_s1.copy().pick_types(eeg=True).plot(
    duration=10,           # 10 secondes affichées
    n_channels=20,         # 20 canaux visibles
    scalings='auto',       # Ajustement auto de l'échelle
    title=f'Sujet 1 - Signal EEG brut (10 secondes)',
    show=False,
    block=False
)
fig1.savefig('../figures/raw_signal_subject1.png', dpi=150)
print("✅ Figure sauvegardée : raw_signal_subject1.png")

# 5.2 Topographie des électrodes (positions 2D)
fig2 = raw_s1.plot_sensors(
    show_names=True,       # Afficher les noms des électrodes
    title='Montage 10-20 - Positions des 64 électrodes',
    show=False
)
fig2.savefig('../figures/topography_64_channels.png', dpi=150)
print("✅ Figure sauvegardée : topography_64_channels.png")

# 5.3 Power Spectral Density (PSD) globale
# Explication : La PSD montre quelle fréquence a quelle puissance
print("\n📈 Calcul de la PSD globale...")

# Calcul sur tout le signal
psd = raw_s1.copy().pick_types(eeg=True).compute_psd(
    fmin=0.5, fmax=50,     # Bande 0.5-50 Hz
    n_fft=2048,            # Taille de la FFT (résolution)
    verbose=False
)

fig3 = psd.plot(
    average=True,          # Moyenne sur tous les canaux
    picks='eeg',
    exclude=[],
    spatial_colors=False,
    show=False
)
fig3.axes[0].set_title('Densité Spectrale de Puissance (PSD) - Moyenne sur 64 canaux')
fig3.axes[0].set_xlabel('Fréquence (Hz)')
fig3.axes[0].set_ylabel('Puissance (dB)')
fig3.axes[0].grid(True, alpha=0.3)
fig3.savefig('../figures/psd_global_subject1.png', dpi=150)
print("✅ Figure sauvegardée : psd_global_subject1.png")

# Mettre en évidence les bandes EEG sur la PSD
plt.figure(figsize=(10, 6))
psd_avg = psd.get_data().mean(axis=0)  # Moyenne sur tous les canaux
freqs = psd.freqs

plt.semilogy(freqs, psd_avg, 'k-', linewidth=1.5)

# Colorer les bandes
bands = [
    ('Delta (0.5-4 Hz)', (0.5, 4), 'lightblue'),
    ('Theta (4-8 Hz)', (4, 8), 'lightgreen'),
    ('Alpha (8-13 Hz)', (8, 13), 'lightcoral'),
    ('Beta (13-30 Hz)', (13, 30), 'lightyellow'),
    ('Gamma (30-50 Hz)', (30, 50), 'plum')
]

for name, (fmin, fmax), color in bands:
    plt.axvspan(fmin, fmax, alpha=0.2, color=color, label=name)

plt.xlabel('Fréquence (Hz)', fontsize=12)
plt.ylabel('Puissance (µV²/Hz)', fontsize=12)
plt.title('PSD avec bandes physiologiques - Sujet 1', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('../figures/psd_with_bands_subject1.png', dpi=150)
plt.show()


In [ ]:
# ============================================================================
# INTERPRÉTATION DE LA PSD
# ============================================================================
# On observe généralement :
# - Pic alpha (8-13 Hz) : repos visuel, yeux ouverts vs fermés
# - Activité beta (13-30 Hz) : motricité et concentration
# - Puissance décroissante 1/f (bruit physiologique)
# - Pic à 50 Hz potentiel (réseau électrique) → sera filtré


TABLEAU RÉCAPITULATIF DES ANNOTATIONS

In [ ]:
print("\n" + "="*60)
print("TABLEAU RÉCAPITULATIF DES ANNOTATIONS")
print("="*60)

# Créer un tableau professionnel
summary_table = []
for subject in subjects:
    raw = raw_data[subject]
    events, event_id = mne.events_from_annotations(raw, verbose=False)

    subject_summary = {
        'Sujet': subject,
        'Durée (s)': raw.times[-1],
        'Nb événements': len(events),
        'T1 (main gauche)': np.sum(events[:, 2] == 1),
        'T2 (main droite)': np.sum(events[:, 2] == 2),
        'T0 (repos)': np.sum(events[:, 2] == 0) if 0 in events[:, 2] else 0
    }
    summary_table.append(subject_summary)

df_summary = pd.DataFrame(summary_table)
print(df_summary.to_string(index=False))

# Sauvegarder le tableau
df_summary.to_csv('../reports/annotations_summary.csv', index=False)
print("\n✅ Tableau sauvegardé : reports/annotations_summary.csv")


SAUVEGARDE POUR LES NOTEBOOKS SUIVANTS

In [ ]:
from pathlib import Path

# Créer les dossiers s'ils n'existent pas
Path("figures").mkdir(exist_ok=True)
Path("reports").mkdir(exist_ok=True)

print("✅ Dossiers créés : figures/, reports/")

In [ ]:
print("\n" + "="*60)
print("SAUVEGARDE INTERMÉDIAIRE")
print("="*60)

# Sauvegarde des objets Raw
for subject in subjects:
    raw_data[subject].save(f'reports/raw_subject_{subject}_preprocessed.fif', 
                           overwrite=True)
    print(f"✅ Sujet {subject} sauvegardé")

# Sauvegarde du résumé JSON (avec conversion explicite des types)
import json
import numpy as np

# Convertir les types numpy en types Python natifs
def convert_to_serializable(obj):
    """Convertit les objets numpy en types JSON compatibles"""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

summary_json = {
    'subjects_loaded': subjects,
    'sampling_frequency': float(raw_s1.info['sfreq']),  # Conversion explicite
    'n_channels': int(len(raw_s1.ch_names)),           # Conversion explicite
    'duration_seconds': float(raw_s1.times[-1]),       # Conversion explicite
    'event_distribution': {k: int(v) for k, v in event_counts.items()}  # Conversion
}

# Sauvegarde
with open('reports/loading_summary.json', 'w') as f:
    json.dump(summary_json, f, indent=2)

print("✅ Rapport JSON sauvegardé : reports/loading_summary.json")

print("\n" + "="*60)
print("NOTEBOOK 01 TERMINÉ AVEC SUCCÈS")
print("="*60)
print("\n📌 RÉSUMÉ DES APPRENTISSAGES :")
print("   1. Format EDF = conteneur médical standard")
print("   2. Objet Raw MNE = DataFrame EEG spécialisé")
print("   3. 64 électrodes suivant montage 10-20")
print("   4. Fréquence 160 Hz = bon compromis")
print("   5. Événements T1 (main gauche) et T2 (main droite)")
print("   6. PSD montre les bandes physiologiques")
print("\n➡️ Prochain notebook : Prétraitement (filtrage + ICA)")
